# Day 32: Profile Attention Memory Growth Across Sequence Lengths

**Layer:** Implementation | **Prerequisite:** Day 30 (SDPA), Day 31 (Flash Attention)


## Concept Overview

Attention memory scales quadratically with sequence length — the 4096-token barrier for large models is fundamentally a memory constraint. This experiment systematically profiles where memory goes as sequence length grows, comparing standard SDPA, FlashAttention, and the KV cache overhead for decode.


In [ ]:
import torch
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import gc, time

print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'Total VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')


## 1. Memory Breakdown by Component


In [ ]:
def attention_memory_breakdown(seq_len, batch, heads, d_head, d_model, num_layers, dtype_bytes=2):
    """Compute memory for each component of attention at inference."""
    T, B, H, d = seq_len, batch, heads, d_head
    D = d_model

    # Weights (constant, loaded once)
    weight_bytes = num_layers * 4 * D * D * dtype_bytes  # Q,K,V,O per layer
    # Activations per layer
    qkv_bytes = 3 * B * T * D * dtype_bytes
    # Attention score matrix (O(T²)) — this is what FlashAttention avoids
    attn_score_bytes = B * H * T * T * dtype_bytes
    # KV cache (O(T))
    kv_cache_bytes = num_layers * 2 * B * H * T * d * dtype_bytes
    # Output
    out_bytes = B * T * D * dtype_bytes

    return {
        'weights_GB': weight_bytes/1e9,
        'kv_cache_GB': kv_cache_bytes/1e9,
        'attn_scores_GB': attn_score_bytes/1e9,
        'activations_GB': (qkv_bytes + out_bytes)/1e9,
    }

# Llama-3-8B config
config = {'heads': 32, 'd_head': 128, 'd_model': 4096, 'num_layers': 32}

print('Attention memory breakdown (Llama-3-8B, batch=1, FP16):')
print(f'{"Seq Len":>10} {"Weights":>10} {"KV Cache":>10} {"Attn Scores":>13} {"Total (no FA)":>15}')
print('-' * 62)
for seq in [512, 1024, 2048, 4096, 8192, 16384, 32768]:
    m = attention_memory_breakdown(seq, batch=1, **config)
    total = sum(m.values())
    print(f'{seq:>10} {m["weights_GB"]:>10.2f} {m["kv_cache_GB"]:>10.2f} {m["attn_scores_GB"]:>13.2f} {total:>15.2f}')
print('(Units: GB)')


## 2. Empirical Memory Profiling


In [ ]:
def profile_attention_memory(seq_len, heads=8, d_head=64, batch=1, use_flash=False, device='cpu'):
    """Measure actual peak memory for attention computation."""
    gc.collect()
    if device == 'cuda':
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()
        baseline = torch.cuda.memory_allocated()

    Q = torch.randn(batch, heads, seq_len, d_head, device=device, dtype=torch.float16)
    K = torch.randn_like(Q)
    V = torch.randn_like(Q)

    if use_flash:
        out = F.scaled_dot_product_attention(Q, K, V, is_causal=True)
    else:
        scale = d_head ** -0.5
        scores = Q @ K.transpose(-2,-1) * scale
        mask = torch.tril(torch.ones(seq_len, seq_len, device=device, dtype=torch.bool))
        scores = scores.masked_fill(~mask, float('-inf'))
        weights = F.softmax(scores, dim=-1)
        out = weights @ V

    if device == 'cuda':
        torch.cuda.synchronize()
        peak = (torch.cuda.max_memory_allocated() - baseline) / 1e6
        return peak
    else:
        return 0  # can't measure CPU peak easily

device = 'cuda' if torch.cuda.is_available() else 'cpu'
seq_lens = [64, 128, 256, 512, 1024] if device == 'cuda' else [64, 128, 256]

results_std = []
results_flash = []
for seq in seq_lens:
    m_std = profile_attention_memory(seq, use_flash=False, device=device)
    m_flash = profile_attention_memory(seq, use_flash=True, device=device)
    results_std.append(m_std)
    results_flash.append(m_flash)
    print(f'seq={seq}: standard={m_std:.1f}MB flash={m_flash:.1f}MB', end='')
    if m_std > 0:
        print(f' reduction={m_std/m_flash:.1f}x')
    else:
        print(' (CPU: no VRAM tracking)')


## 3. Plotting Growth Curves


In [ ]:
# Analytical curves
seq_range = np.array([128, 256, 512, 1024, 2048, 4096, 8192, 16384])
heads, d_head, batch = 32, 128, 1

# Attention score matrix: B*H*T*T*2 bytes
attn_scores_mb = batch * heads * seq_range**2 * 2 / 1e6
# KV cache: 2*32*H*T*d*2 bytes (32 layers)
kv_cache_mb = 2 * 32 * heads * seq_range * d_head * 2 / 1e6

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

ax1.loglog(seq_range, attn_scores_mb, 'r-o', label='Attention scores O(n²)')
ax1.loglog(seq_range, kv_cache_mb, 'b-s', label='KV cache O(n)')
ax1.set_xlabel('Sequence Length'); ax1.set_ylabel('Memory (MB)')
ax1.set_title('Attention Memory: Scores vs KV Cache')
ax1.legend(); ax1.grid(True)

# The crossover point
crossover = np.argmin(np.abs(attn_scores_mb - kv_cache_mb))
ax1.axvline(seq_range[crossover], color='gray', linestyle='--',
            label=f'Crossover at seq={seq_range[crossover]}')
ax1.legend()

# Model total memory at different batch sizes
ax2.set_title('Total Attention Memory vs Batch Size\n(seq=2048, Llama-3-8B)')
batches = [1, 2, 4, 8, 16, 32]
for seq in [1024, 2048, 4096]:
    mems = [batch * heads * seq**2 * 2 / 1e9 for batch in batches]  # GB
    ax2.plot(batches, mems, '-o', label=f'seq={seq}')
ax2.set_xlabel('Batch Size'); ax2.set_ylabel('Attention Score Memory (GB)')
ax2.legend(); ax2.grid(True)

plt.tight_layout()
plt.savefig('attention_memory_profile.png', dpi=100, bbox_inches='tight')
plt.show()


## Experiments

1. Measure prefill time vs sequence length. Does it scale O(n²) empirically?
2. Profile the KV cache growth in the decoder loop from Day 29.
3. Calculate the maximum sequence length that fits in your GPU VRAM for Llama-3-8B without FlashAttention vs with.


## Key Takeaways

- Attention score matrix is O(n²) in memory: 4096² × 32 heads × FP16 = 1GB just for scores, before values and output.
- KV cache is O(n) per layer: Llama-3-8B at seq=4096 needs ~2GB for KV cache, growing linearly.
- The crossover where attention scores exceed KV cache happens around seq=1K-4K depending on head count.
- FlashAttention converts the O(n²) score matrix from HBM to on-chip SRAM tiles — this is the reason it enables 100K+ context lengths.

## References
- Dao et al. (2022), "FlashAttention"
- Pope et al. (2022), "Efficiently Scaling Transformer Inference"
